OpenAI의 assistant에 해당하는 역할 이름은 Gemini에서 model입니다.

직접 리스트를 관리하기보다 chat = client.chats.create()를 만들고 chat.send_message()를 사용하는 것이 코드가 훨씬 간단하고 에러 방지에도 좋습니다.

In [ ]:
#!uv add  openai

In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# 0.
load_dotenv()

# 1. 클라이언트 초기화 (OPENAI_API_KEY 환경변수를 자동으로 읽어옵니다)
client = OpenAI()

# 2. 대화 기록(History)을 저장할 리스트 생성 및 페르소나 설정
messages = [
    {
        "role": "system",
        "content": "당신은 친절하고 능숙한 AI 어시스턴트입니다. 사용자 질문에 정확하고 명쾌하게 답변해주세요.",
    }
]

print("=== OpenAI Chatbot 시작 ('exit' 또는 'quit' 입력 시 종료) ===")

while True:
    # 사용자 입력 받기
    user_input = input("\n사용자: ").strip()

    # 종료 조건 체크
    if user_input.lower() in ["exit", "quit", "종료"]:
        print("챗봇을 종료합니다.")
        break

    if not user_input:
        continue

    # 사용자 메시지를 대화 기록에 추가
    messages.append({"role": "user", "content": user_input})

    try:
        # API 호출 (대화 맥락 전체 전달)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0.7,
        )

        # 모델 응답 추출 및 출력
        bot_response = response.choices[0].message.content
        print(f"\nAI: {bot_response}")

        # 모델 응답도 대화 기록에 추가 (다음 턴에서 맥락 유지)
        messages.append({"role": "assistant", "content": bot_response})

    except Exception as e:
        print(f"\n오류가 발생했습니다: {e}")

=== OpenAI Chatbot 시작 ('exit' 또는 'quit' 입력 시 종료) ===



오류가 발생했습니다: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************t7oA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
챗봇을 종료합니다.


In [ ]:
#!uv add google-api

In [ ]:
import os
from google import genai
from google.genai import types

client = genai.Client()

config = types.GenerateContentConfig(
    system_instruction="당신은 친절한 AI 어시스턴트입니다.",
    temperature=0.7,
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=config
)

print("=== Gemini Streaming Chatbot 시작 ('exit' 입력 시 종료) ===")

while True:
    user_input = input("\n사용자: ").strip()

    if user_input.lower() in ["exit", "quit", "종료"]:
        print("챗봇을 종료합니다.")
        break

    if not user_input:
        continue

    try:
        # send_message_stream 사용
        response = chat.send_message_stream(user_input)     # 문맥이 자동 유지. 별도로 넘겨주는거 없음?

        print("\nAI: ", end="", flush=True)
        for chunk in response:
            print(chunk.text, end="", flush=True)
        print()  # 줄바꿈

    except Exception as e:
        print(f"\n오류가 발생했습니다: {e}")

=== Gemini Streaming Chatbot 시작 ('exit' 입력 시 종료) ===

AI: 안농! 👋 반가워요!

무슨 일로 오셨어요? 아니면 그냥 인사하러 오신 건가요? 😊

AI: 
오류가 발생했습니다: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 14.325294294s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateReque